# Linear / Logistic / Softmax — pure NumPy 🧮

Three from-scratch models in one notebook, **NumPy only** (no sklearn, no torch):

| Section | Model | What it does |
|---|---|---|
| 1 | **Linear Regression** | Fits a line/hyperplane to *any amount* of data — 1 sample or 1,000,000. Closed-form solution + gradient descent. |
| 2 | **Logistic Regression** | Binary classification via the sigmoid. Gradient descent, decision boundary, accuracy. |
| 3 | **Softmax Regression** | Multi-class classification (one-vs-all weight vectors + softmax probabilities). Works on any number of classes. |

Everything is vectorized, so the same code handles `N = 1` row or `N = 10^6` rows, and `D = 1` feature or `D = 50` features.

## Setup (numpy only)

In [1]:
import numpy as np

rng = np.random.default_rng(42)   # reproducible randomness everywhere below
np.set_printoptions(precision=4, suppress=True)

OpenBLAS WARNING - could not determine the L2 cache size on this system, assuming 256k


## 1. Linear Regression (any amount of data)

Model: $\hat{y} = Xw + b$, loss $= \frac{1}{N}\sum (\hat{y}-y)^2$.

Two ways to fit:
- **Closed form (normal equation):** $w = (X^TX)^{-1}X^Ty$ — exact, one shot.
- **Gradient descent:** iterative, scales to huge datasets.

Both accept *any* number of samples `N` and features `D`.

In [2]:
class LinearRegression:
    """Simple linear regression with numpy. Handles ANY number of samples/features."""

    def __init__(self, lr=0.1, iters=1000):
        self.lr, self.iters = lr, iters
        self.w = None
        self.b = 0.0
        self.loss_history = []

    @staticmethod
    def _prep(X, y):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:                       # 1-D input -> single feature column
            X = X.reshape(-1, 1)              # works even for ONE sample: [x] -> [[x]]
        y = np.asarray(y, dtype=float).reshape(-1)
        if len(X) != len(y):
            raise ValueError(f"N mismatch: X has {len(X)} rows, y has {len(y)}")
        return X, y

    def fit(self, X, y):
        X, y = self._prep(X, y)
        N, D = X.shape                        # any amt of data lol
        # standardize features so plain GD converges fast for any scale/size
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-12
        Xs = (X - self.mu) / self.sd
        self.y_mu, self.y_sd = y.mean(), y.std() + 1e-12
        ys = (y - self.y_mu) / self.y_sd

        w, b = np.zeros(D), 0.0
        self.loss_history = []
        for _ in range(self.iters):
            pred = Xs @ w + b
            err = pred - ys
            self.loss_history.append(np.mean(err ** 2))
            grad_w = 2 * (Xs.T @ err) / N     # vectorized gradient -> fine for huge N
            grad_b = 2 * err.mean()
            w -= self.lr * grad_w
            b -= self.lr * grad_b

        # convert standardized-space params back to ORIGINAL units:
        # y = ((X-mu)/sd) @ w + b  (in y-standardized units)  =>  y = X @ W + B
        self.w = w * self.y_sd / self.sd
        self.b = b * self.y_sd + self.y_mu - (self.mu * self.w).sum()
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return X @ self.w + self.b

    @staticmethod
    def closed_form(X, y):
        """Exact least squares: w = (X^T X)^-1 X^T y  (bias folded in as a ones column)."""
        X, y = LinearRegression._prep(X, y)
        Xb = np.hstack([np.ones((len(X), 1)), X])          # first column = bias
        theta = np.linalg.lstsq(Xb, y, rcond=None)[0]      # lstsq = stable inverse
        return theta[0], theta[1:]                         # bias, weights

    def score(self, X, y):                                 # R^2
        X, y = self._prep(X, y)
        ss_res = ((self.predict(X) - y) ** 2).sum()
        ss_tot = ((y - y.mean()) ** 2).sum() + 1e-12
        return 1 - ss_res / ss_tot

In [3]:
# --- demo data: y = 3.5x - 2 + noise, generated for ANY N you want ---
def make_regression_data(N=200, D=1, seed=0):
    r = np.random.default_rng(seed)
    X = r.uniform(-3, 3, size=(N, D))
    true_w = np.array([3.5, -1.2, 0.8, 2.1, -0.5])[:D]
    y = X @ true_w + 2.0 + r.normal(0, 1.0, size=N)
    return X, y

Xtr, ytr = make_regression_data(N=200)

gd = LinearRegression(lr=0.1, iters=1000).fit(Xtr, ytr)
print("GD   -> bias ≈ {:.3f}, weight ≈ {:.3f}, R² = {:.4f}".format(gd.b, gd.w[0], gd.score(Xtr, ytr)))

b_cf, w_cf = LinearRegression.closed_form(Xtr, ytr)
print("CF   -> bias ≈ {:.3f}, weight ≈ {:.3f}".format(b_cf, w_cf[0]))
print("loss history (first/last):", gd.loss_history[0], "->", round(gd.loss_history[-1], 5))

GD   -> bias ≈ 1.921, weight ≈ 3.458, R² = 0.9740
CF   -> bias ≈ 1.921, weight ≈ 3.458
loss history (first/last): 0.9999999999996845 -> 0.02597


In [4]:
# --- proof it really takes ANY amount of data ---
for N in [1, 2, 3, 10, 1_000, 100_000, 1_000_000]:
    Xn, yn = make_regression_data(N=N)
    m = LinearRegression(lr=0.1, iters=300).fit(Xn, yn)
    print(f"N={N:>9}  ->  w={m.w[0]:.3f}, b={m.b:.3f}, final MSE={m.loss_history[-1]:.4f}")

# --- and any number of features (multi-variate) ---
Xm, ym = make_regression_data(N=500, D=5)
mm = LinearRegression(lr=0.1, iters=2000).fit(Xm, ym)
print("\n5-feature fit:", mm.w, "| intercept ≈ 2.0 ->", round(mm.b, 3), "| R² =", round(mm.score(Xm, ym), 4))

# --- one single data point? no dof to fit -> falls back to predicting y itself ---
one = LinearRegression(iters=200).fit([2.0], [5.0])
print("single-point prediction at x=2:", one.predict([2.0]))

# --- tiny 2-point exact line: gradient descent should match the closed form ---
two = LinearRegression(iters=8000, lr=0.2).fit([[1.0], [3.0]], [5.0, 13.0])   # true: y = 4x + 1
b_t, w_t = LinearRegression.closed_form([[1.0], [3.0]], [5.0, 13.0])
print("2-point GD :", two.w[0], two.b, "| closed form:", w_t, b_t)

N=        1  ->  w=0.000, b=4.744, final MSE=0.0000
N=        2  ->  w=3.743, b=2.441, final MSE=0.0000
N=        3  ->  w=3.462, b=1.935, final MSE=0.0053
N=       10  ->  w=3.657, b=1.501, final MSE=0.0134
N=     1000  ->  w=3.497, b=1.988, final MSE=0.0287


N=   100000  ->  w=3.501, b=2.003, final MSE=0.0266


N=  1000000  ->  w=3.499, b=2.001, final MSE=0.0264

5-feature fit: [ 3.4758 -1.2292  0.7734  2.1076 -0.5075] | intercept ≈ 2.0 -> 1.91 | R² = 0.9848
single-point prediction at x=2: [5.]


2-point GD : 3.999999999999999 1.0000000000000018 | closed form: [4.] 0.9999999999999963


## 2. Logistic Regression (binary classification)

Model: $P(y=1|x) = \sigma(x^Tw+b)$ with sigmoid $\sigma(z)=\frac{1}{1+e^{-z}}$.

Loss: binary cross-entropy $=-\frac{1}{N}\sum[y\log p + (1-y)\log(1-p)]$, minimized with gradient descent. Again: any `N`, any `D`.

In [5]:
def sigmoid(z):
    z = np.clip(z, -500, 500)          # avoid overflow in exp for extreme inputs
    return 1.0 / (1.0 + np.exp(-z))


class LogisticRegression:
    """Binary logistic regression, numpy only, any amount of data."""

    def __init__(self, lr=0.1, iters=1000, l2=0.0):
        self.lr, self.iters, self.l2 = lr, iters, l2
        self.loss_history = []

    @staticmethod
    def _prep(X, y):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        y = np.asarray(y, dtype=float).reshape(-1)
        if not np.isin(y, (0, 1)).all():
            raise ValueError("labels must be 0/1")
        return X, y

    def fit(self, X, y):
        X, y = self._prep(X, y)
        N, D = X.shape
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-12
        Xs = np.hstack([(X - self.mu) / self.sd, np.ones((N, 1))])  # bias as extra col
        self.theta = np.zeros(D + 1)

        self.loss_history = []
        for _ in range(self.iters):
            p = sigmoid(Xs @ self.theta)
            err = p - y                                  # beautiful simple gradient
            self.loss_history.append(-np.mean(y * np.log(p + 1e-12) +
                                              (1 - y) * np.log(1 - p + 1e-12)))
            grad = 2 * Xs.T @ err / N + 2 * self.l2 * self.theta
            self.theta -= self.lr * grad
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        Xs = np.hstack([(X - self.mu) / self.sd, np.ones((len(X), 1))])
        return sigmoid(Xs @ self.theta)                  # P(y=1 | x)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def score(self, X, y):
        X, y = self._prep(X, y)
        return (self.predict(X) == y).mean()

In [6]:
# --- demo: two gaussian blobs ---
def make_blobs(N=400, seed=1):
    r = np.random.default_rng(seed)
    c0 = r.normal(loc=[-1.5, -1.5], scale=0.8, size=(N // 2, 2))
    c1 = r.normal(loc=[1.5, 1.5], scale=0.8, size=(N - N // 2, 2))
    X = np.vstack([c0, c1])
    y = np.concatenate([np.zeros(len(c0)), np.ones(len(c1))])
    idx = r.permutation(len(X))
    return X[idx], y[idx]

Xtr, ytr = make_blobs()
clf = LogisticRegression(lr=0.1, iters=1000).fit(Xtr, ytr)
print("weights:", clf.theta[:-1].round(3), "| bias:", round(clf.theta[-1], 3))
print("train accuracy:", clf.score(Xtr, ytr))
print("loss history (first/last): {:.4f} -> {:.4f}".format(clf.loss_history[0], clf.loss_history[-1]))

probs = clf.predict_proba(Xtr[:5])
preds = clf.predict(Xtr[:5])
print("\nfirst 5 probas:", probs.round(3), "-> preds:", preds, "| true:", ytr[:5].astype(int))

weights: [3.537 3.572] | bias: 0.455
train accuracy: 0.995
loss history (first/last): 0.6931 -> 0.0216

first 5 probas: [1.    0.999 1.    1.    0.003] -> preds: [1 1 1 1 0] | true: [1 1 1 1 0]


In [7]:
# --- any amount of data again ---
for N in [1, 5, 100, 10_000, 500_000]:
    Xn, yn = make_blobs(N=N)
    c = LogisticRegression(lr=0.1, iters=300).fit(Xn, yn)
    print(f"N={N:>7} -> acc={c.score(Xn, yn):.3f}, BCE={c.loss_history[-1]:.4f}")

N=      1 -> acc=1.000, BCE=0.0174
N=      5 -> acc=1.000, BCE=0.0121
N=    100 -> acc=1.000, BCE=0.0208


N=  10000 -> acc=0.996, BCE=0.0347


N= 500000 -> acc=0.996, BCE=0.0359


## 3. Softmax Regression (multi-class)

Also called *multinomial logistic regression*. For $K$ classes:

$$P(y=k \mid x) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}, \quad z_k = x^T W_k + b_k$$

Trained with cross-entropy + gradient descent. The gradient is famously tidy: $\nabla_W = X^T(P - Y_{onehot})/N$. Handles **any number of classes and samples**.

In [8]:
def softmax(Z):
    """Numerically stable softmax over the last axis (rows are independent)."""
    Z = Z - Z.max(axis=1, keepdims=True)     # shift by max -> no exp overflow
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)


class SoftmaxRegression:
    """Multi-class softmax regression, numpy only, any K classes / N samples."""

    def __init__(self, lr=0.1, iters=1000, l2=0.0):
        self.lr, self.iters, self.l2 = lr, iters, l2
        self.loss_history = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        y = np.asarray(y, dtype=int).reshape(-1)
        N, D = X.shape
        self.classes = np.unique(y)
        K = len(self.classes)
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        yi = np.array([self.class_to_idx[c] for c in y])

        self.mu, self.sd = X.mean(0), X.std(0) + 1e-12
        Xs = np.hstack([(X - self.mu) / self.sd, np.ones((N, 1))])   # bias column

        # one-hot targets
        Y = np.zeros((N, K))
        Y[np.arange(N), yi] = 1.0

        self.W = np.zeros((D + 1, K))
        self.loss_history = []
        for _ in range(self.iters):
            P = softmax(Xs @ self.W)                      # (N, K) probabilities
            self.loss_history.append(-np.log(P[np.arange(N), yi] + 1e-12).mean())
            grad = Xs.T @ (P - Y) / N + 2 * self.l2 * self.W
            self.W -= self.lr * grad
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        Xs = np.hstack([(X - self.mu) / self.sd, np.ones((len(X), 1))])
        return softmax(Xs @ self.W)

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes[proba.argmax(axis=1)]

    def score(self, X, y):
        return (self.predict(X) == np.asarray(y).reshape(-1)).mean()

In [9]:
# --- demo: 3-class spiral-ish blobs (any K!) ---
def make_classes(N_per=150, K=3, D=2, seed=2):
    r = np.random.default_rng(seed)
    X, y = [], []
    for k in range(K):
        center = np.array([[2.5 * np.cos(2*np.pi*k/K), 2.5 * np.sin(2*np.pi*k/K)]])[0, :D]                  if D == 2 else r.normal(size=D)
        X.append(r.normal(center, 0.7, size=(N_per, D)))
        y.append(np.full(N_per, k))
    return np.vstack(X), np.concatenate(y)

Xtr, ytr = make_classes(N_per=150, K=3)
sm = SoftmaxRegression(lr=0.1, iters=1000).fit(Xtr, ytr)
print("classes found:", sm.classes, "| train accuracy:", sm.score(Xtr, ytr))
print("loss history: {:.4f} -> {:.4f}".format(sm.loss_history[0], sm.loss_history[-1]))

# probability rows sum to 1? sanity check
p = sm.predict_proba(Xtr[:3])
print("\nprobabilities for 3 points:\n", p)
print("row sums:", p.sum(axis=1))

classes found: [0 1 2] | train accuracy: 1.0
loss history: 1.0986 -> 0.0255

probabilities for 3 points:
 [[0.9962 0.001  0.0028]
 [0.9518 0.0004 0.0478]
 [0.9996 0.0004 0.    ]]
row sums: [1. 1. 1.]


In [10]:
# --- works with ANY number of classes too ---
for K in [2, 3, 5, 10, 26]:
    Xk, yk = make_classes(N_per=80, K=K)
    m = SoftmaxRegression(lr=0.1, iters=800).fit(Xk, yk)
    print(f"K={K:>2} classes -> acc={m.score(Xk, yk):.3f}, final CE loss={m.loss_history[-1]:.4f}")

K= 2 classes -> acc=1.000, final CE loss=0.0125
K= 3 classes -> acc=0.996, final CE loss=0.0326


K= 5 classes -> acc=0.963, final CE loss=0.2056


K=10 classes -> acc=0.721, final CE loss=0.9447


K=26 classes -> acc=0.330, final CE loss=2.1027


## Wrap-up ✅

- **LinearRegression**: normal equation + gradient descent, arbitrary `N × D`, reports R².
- **LogisticRegression**: sigmoid + BCE gradient descent, probabilities & thresholded predictions.
- **SoftmaxRegression**: stable softmax, one-hot cross-entropy, arbitrary number of classes.

All three share the same tricks: auto-shape handling (1-D → column), feature standardization (so fixed-LR GD converges for any data scale), numerically stable `sigmoid`/`softmax`, and fully vectorized gradients so 1,000,000-row fits run in seconds. Zero dependencies besides NumPy. 🎉